# 🌍 Macroe Folder - Complete Demonstration

This notebook demonstrates the **refactored** macroeconomic analysis tools:
1. **Currency Strength Meter** - Ranks currencies by macro sentiment
2. **Polymarket Client** - Safe trading interface with retry logic
3. **Global Pipeline Runner** - Multi-region data collection

## ✅ What Was Fixed
- ✅ Type safety (full mypy compliance)
- ✅ Proper error handling (no more silent failures)
- ✅ Eliminated hardcoded credentials
- ✅ Class-based architecture (no more spaghetti code)
- ✅ Async/await patterns instead of threading

## 📦 Setup & Imports

In [ ]:
import os
import sys
import asyncio
from pathlib import Path

# Add macroe folder to path
macroe_path = Path.cwd() if Path.cwd().name == 'macroe' else Path.cwd() / 'macroe'
sys.path.insert(0, str(macroe_path))

# Verify we can import refactored modules
try:
    from currency_strength_meter_refactored import CurrencyStrengthMeter
    from polymarket_client import PolymarketClient
    print("✅ All modules imported successfully")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print(f"Current directory: {Path.cwd()}")
    print(f"Macroe path: {macroe_path}")

## 🔧 Demo 1: Currency Strength Meter

Analyzes macro data files (generated by `global_runner.py`) and ranks currencies.

In [ ]:
# Initialize the meter (points to macroe folder for CSV files)
meter = CurrencyStrengthMeter(data_dir=macroe_path)

# Calculate strength scores
scores = meter.calculate_strength()

# Display results in rich table
meter.display_results(scores)

### 📊 Accessing Raw Data

The refactored version returns structured data for programmatic use:

In [ ]:
if scores:
    print("Top 3 Strongest Currencies:")
    for i, score in enumerate(scores[:3], 1):
        print(f"{i}. {score['Currency']}: {score['Total_Score']} (avg: {score['Avg_Score']})")
    
    # Example: Trading signal
    strongest = scores[0]['Currency']
    weakest = scores[-1]['Currency']
    print(f"\n💡 Trading Idea: Long {strongest}, Short {weakest}")
else:
    print("No data available. Run global_runner.py first.")

## 🔐 Demo 2: Polymarket Client (Safe Mode)

**Note:** This requires valid credentials in `.env` file.
Demo shows the improved API without executing real trades.

In [ ]:
# Check if credentials exist
has_credentials = os.getenv('PRIVATE_KEY') is not None

if has_credentials:
    try:
        # Initialize client (will fail gracefully if credentials invalid)
        client = PolymarketClient()
        print("✅ Polymarket client initialized")
        print(f"Proxy address: {client.proxy_address}")
    except Exception as e:
        print(f"⚠️ Client initialization failed: {e}")
        client = None
else:
    print("ℹ️ No credentials found. Skipping Polymarket demo.")
    print("To enable: Create .env file with PRIVATE_KEY and POLYMARKET_PROXY")
    client = None

### 🔍 Scan for Active Markets (if credentials available)

In [ ]:
import aiohttp

async def scan_demo():
    if client is None:
        print("Client not initialized. Skipping market scan.")
        return
    
    async with aiohttp.ClientSession() as session:
        print("Scanning for active 15m crypto markets...")
        markets = await client.scan_15min_markets(session)
        
        if markets:
            print(f"\n✅ Found {len(markets)} active markets:")
            for m in markets:
                print(f"  • {m.slug} (ends: {m.end_time})")
        else:
            print("No active markets found")

# Run async function in Jupyter
await scan_demo()

## 🧪 Demo 3: Helper Functions

Testing utility functions from the refactored code.

In [ ]:
from datetime import datetime, timezone

# Test window calculation
current_window = PolymarketClient.get_15min_window_epoch(0)
next_window = PolymarketClient.get_15min_window_epoch(1)

print(f"Current 15m window: {datetime.fromtimestamp(current_window, tz=timezone.utc)}")
print(f"Next 15m window:    {datetime.fromtimestamp(next_window, tz=timezone.utc)}")
print(f"\nWindow size: {(next_window - current_window) / 60} minutes")

## 📈 Integration Example: Macro + Trading

Combining currency strength with market scanning for informed trading.

In [ ]:
def generate_trading_signal(currency_scores, active_markets):
    """
    Example: Use macro sentiment to filter crypto trades.
    
    Logic: Only trade crypto if USD sentiment is neutral-to-bullish
    (strong USD often correlates with crypto weakness)
    """
    if not currency_scores or not active_markets:
        return None
    
    # Find USD score
    usd_data = next((s for s in currency_scores if s['Currency'] == 'USD'), None)
    
    if usd_data is None:
        return None
    
    usd_score = usd_data['Total_Score']
    
    # Simple logic: avoid crypto longs when USD very strong
    if usd_score > 20:
        return {
            'action': 'AVOID',
            'reason': f'USD too strong (score: {usd_score})',
            'markets': []
        }
    else:
        return {
            'action': 'CONSIDER',
            'reason': f'USD neutral/weak (score: {usd_score})',
            'markets': active_markets
        }

# Demo
print("Example Trading Signal Generator:")
print("-" * 50)
if scores:
    signal = generate_trading_signal(scores, [])
    if signal:
        print(f"Action: {signal['action']}")
        print(f"Reason: {signal['reason']}")
else:
    print("No currency data available")

## 🎯 Summary

This notebook demonstrated:

1. ✅ **Currency Strength Meter** - Production-ready with type safety
2. ✅ **Polymarket Client** - Reusable, async, error-tolerant
3. ✅ **Integration Pattern** - Combining macro + market data

### Old vs New

| Aspect | Before | After |
|--------|--------|-------|
| Type Safety | ❌ No hints | ✅ Full typing |
| Error Handling | ❌ Silent failures | ✅ Logged + recoverable |
| Code Reuse | ❌ Duplicated | ✅ Modular classes |
| Testability | ❌ Global state | ✅ Dependency injection |
| Security | ❌ Hardcoded creds | ✅ Environment vars |

### Next Steps

- Run `global_runner.py` to populate currency data
- Add `.env` file with Polymarket credentials for live trading
- Integrate with `quant_terminal` backend for unified dashboard